# Decoder
**Decoder** architectures are extremely similar to encoder models. Both models utilize the transformer architecture to capture semantic relationships and meanings within sequences. The main difference is in the goals. Unlike encoders, decoders aim to perform next token prediction, which means that the outputs represent a distribution over the next word in the seqence. More formally, the objective can be formulated using the autoregressive model, which we covered in the language modeling notebook. Consider the sequence "The chicken crossed the road." Factoring joint probabilities over the entire sequence, we can decompose the probability of the sequence as
$$ 
\begin{align*}
\mathbb{P}(\text{The chicken crossed the road}) &= \mathbb{P}(\text{The})\\
&\quad \times\mathbb{P}(\text{chicken} | \text{The}) \\
&\quad \times\mathbb{P}(\text{crossed} | \text{The chicken}) \\
&\quad \times\mathbb{P}(\text{the} | \text{The chicken crossed}) \\
&\quad \times\mathbb{P}(\text{road} | \text{The chicken crossed the})
\end{align*}
$$

## Pre-training
In a similar vein to encoders, the output of decoders can be expressed as
$$ H = g_{\hat{\theta}}(X) = \text{Decode}_{\hat{\theta}}(X) $$
We again combine the decoder with a softmax output layer to receive supervision signals.
$$
\begin{align}
    \text{Softmax}_W\left[\text{Decode}_{\hat{\theta}}(X)\right] &= \text{Softmax}_W\left[H\right]\\
    &= \text{Softmax}\left[H W\right]\\
        &= \begin{bmatrix}
           p^{W, \theta}_1 \\
           \vdots \\
           p^{W, \theta}_N
         \end{bmatrix}
\end{align}
$$

Up until this point, the procedure for decoders is exactly the same as that of encoders. However, a key difference is how we interpret these softmax probabilities. For decoders, this output actually represents a distribution over the underlying vocabulary. In other words, it asks: "given the sequence that was inputed, what is the distribution over the next possible tokens?" This should be reflected in the probabilities themselves, where
$$p^{W, \theta}_i = \mathbb{P}(x_{i+1} = k | x_0, ..., x_i; W, \theta) \in  \mathbb{R}^{|V|}$$
It is worth emphasizing that **each softmax output should be over the entire vocabulary**, which is quite distinct from the interpretation of encoders (although models like BERT have similar dimensions for their outputs). This means that the linear map $W$ should transform the embeddings to the size of the vocabulary $|V|$ (i.e. $W \in \mathbb{R}^{D\times |V|}$). This ensures that the softmax layer should have the right dimensions $\text{Softmax}\left[H W\right]\in \mathbb{R}^{(N+1) \times |V|}$.

Let's first look at how the pre-training set up differs to accomadate decoder objectives.

### Masking
In true self-supervised learning fashion, the pre-training procedure requires us to select a subset of tokens in the sequence to use for training. The model is run on given text examples. Then the model's weights are adjusted to either reward or punish it based on the probability it assigns to the true next word. In an ideal world, we want to use the entire sequence for training. But, training the models over the entire sequence allows predictions at earlier predictions to "peek into the future" and cheat, which skews evaluation heavily. The solution, much like encoders, is to use a masking scheme -- although done a slightly different manner. To provide relevant training samples in a self-supervised fashion, we can simply use the $i$-th token as the label for the sequence from $x_0$ to $x_{i-1}$. 

A key observation here is that tokens only interact in the self-attention layers in a transformer network. Hence, the problem can be resolved by ensuring that the attention to the answer and the right context is zero. To do this, we set the corresponding dot products in the self-attention computation to $-\infty$ before they are passed through the softmax function in the attention layer.

![masking](masking.png)

We can now combine everything to construct a loss function. Given a sequence of tokens $x_1, \cdots, x_N$, the autoregressive model tells us that the product of conditional densities of the sequences can be expressed in terms of the joint probability over the entire sequence. Thus, finding the optimal probabilities for each token conditioned on each subsequence (i.e. $\mathbb{P}(x_{i+1} = k | x_0, ..., x_i; W, \theta)$) is equivalent to finding the optimal joint probability overall. We then have the objective

$$\argmax_{W, \theta} \prod_{i=1}^N p^{W, \theta}_i \iff \argmax_{W, \theta} \sum_{i=1}^N \log p^{W, \theta}_i$$

and so the log loss function is

$$L = -\sum^N_{i=1} \log p^{W, \theta}_i = -\sum^N_{i=1} \log \mathbb{P}(x_{i+1} = k | x_0, ..., x_i)$$

which can also be expressed as the cross-entropy loss

$$L = -\sum^N_{i=1} \sum^{|V|}_{v=1}y_{i, v}\log p^{W, \theta}_i$$

Another important thing to note is that because a single sequence contains many prefixes, the model can be trained on all their next-token predictions in parallel. Instead of learning from just one “what comes next?” example per sequence, it learns from every position at once, turning a single sentence into dozens or hundreds of supervised training signals.

## Sampling/decoding strategies
It follows that the next logical step is to output the probability with the highest probability. But we are getting ahead of ourselves. Upon completion of training, we obtain a distribution over the entire vocabulary for the next word in the sequence. The distribution provides us with a way to sample words in the vocabulary, but a question remains: "how exactly should we sample?" The most intuitive answer is just to pick the word with the highest probability - i.e. **greedy sampling**. A nice property of greedy search is effiency. Suppose we want a fixed sequence of $K$ tokens as the output and the vocabulary size is $|V|$, the total number of sequences that that can be outputed is $O(|V|^K)$, which is prohibitively large. Greedy search, in contrast, finds the highest probability word given at each point of the output, which leads to a $O(|V|K)$ runtime. Pretty solid.

However, picking the highest probability word every time leads to very boring answers in which common words are picked every time. If you used earlier versions of ChatGPT, you probably remember how formulaic and predictable responses to questions might be (even if greedy sampling wasn't used).

To address this issue, there a several other sampling strategies that can be employed.
- **Beam Search**: Instead of choosing the single most probable token value at each step, we maintain a set of $B$ hypotheses, where $B$ is called the **beam width**, each consisting of a sequence of token values up to step $n$. We then feed all these sequences through the network, and for each sequence we find the $B$ most probable token values, thereby creating $B^2$ possible hypotheses for the extended sequence. This list is then pruned by selecting the most probable $B$ hypotheses according to the total probability of the extended sequence. Thus, the beam search algorithm maintains $B$ alternative sequences and keeps track of their probabilities, finally selecting the most probable sequence amongst those considered. 

    Intuitively, instead of choosing a single best word (as in greedy search), we choose $B$ possible best words. Also, instead of choosing words independent of position, we take $B$ possible sequences (hence, $B^2$) and we consider the probabilities of the combination of all preceding along with word in the current position. So, we are essentially searching through a "lightbeam" of possibilities.

    Because the probability of a sequence is obtained by multiplying the probabilities at each step of the sequence and since these probability are always less than or equal to one, a long sequence will generally have a lower probability than a short one, biasing the results towards short sequences. Thus, we want to normalize the sequences by the corresponding lengths. Beam search has cost $O(BKN)$, which is again linear in the sequence length. However, the cost of generating a sequence is increased by a factor of $B$, and so for very large language models, where the cost of inference can become significant, this makes beam search much less attractive.
- **Temperature Sampling**: As mentioned above, greedy sampling can lead to generic or sometimes nonsensical sentences. Beam search aims to resolve this by working at the sequence level. Another approach is to use a method called temperature sampling, which still operates on the token level. The idea behind temperature is to redistribute the probabilities of possible values to increase the probability of rarer tokens, enabling models to create more creative responses. For the probability of the $i$-th token, we have
    $$ y_i = \frac{\exp (a_i / T)}{\sum_j exp(a_j/T)}$$
    where $a_i$ are the outputs at the final layer activations (logit vectors). Here, when $T\rightarrow0$, the probability mass is concentrated at the most probably state. When $T=1$, we get the regular softmax function. As $T\rightarrow \infty$, the softmax converges to a uniform distribution. Thus, by choosing $0<T<1$, the probabilities concentrates in high probability states.

    To see an example, consider a two token example, where the probability of token a is $0.13$ and the probability of token b is $0.87$ for the regular softmax ($T=1$). For $T=0.5$, then the probabilities might change to $0.02$ and $0.98$, respectively. On the other hand, for $T=2$, the probabilities might change to something like $0.27$ and $0.73$, respectively.
- **Top-k Sampling**: Top-k sampling is a strategy used to reduce the computational workload of sampling. After logits are computed, we pick the top $k$ logits and perform the softmax over these logits. Depending on how diverse you want your application to be, k can be anywhere from 50 to 500, much smaller than a model’s vocabulary size. The model then samples from these top values. A smaller k value makes the text more predictable but less interesting, as the model is limited to a smaller set of likely words.
- **Top-p Sampling (Nucleus sampling)**: In top-k sampling, the number of candidates, $k$, is fixed. But this depending on the context, we might need to consider more or less candidates in a dynamic fashion. For example, the question "Are you hungry?" and "How hungry are you?" might elicit drastically different responses. The former is more or less a yes/no question whereas the latter is more open-ended. Top-p, also known as nucleus sampling, allows for a more dynamic selection of values to be sampled from. In top-p sampling, the model sums the probabilities of the most likely next values in descending order and stops when the sum reaches p. Only the values within this cumulative probability are considered. Common values for top-p (nucleus) sampling in language models typically range from $0.9$ to $0.95$. Unlike top-k, top-p doesn’t necessarily reduce the softmax computation load. Its benefit is that because it focuses on only the set of most relevant values for each context, it allows outputs to be more contextually appropriate.

## Prompting
Unlike encoder models, which treat the model as a component on which downstream tasks can be performed, in generative AI, LLMs are considered complete systems and are employed to address NLP problems by making use of their generation nature.

More formally, let $x_0, x_1, ..., x_m$ be the sequence given by the user (prompt) and let $y_1, ..., y_n$ be the sequence generated after the context is given. Then, the inference of LLMs can be defined as the problem of finding the most likely sequence $y$ given $x$:
$$
\begin{align*}
\hat{y} &= \arg\max_y log \mathbb{P}(y|x) \\
&= \arg\max_y\sum^n_{i=1}\log \mathbb{P}(y_i|x_0, ..., x_m, y_0, ..., y_{i-1})
\end{align*}
$$

For example, we might have something like
$$
\begin{align*}
&\quad\text{John seems happy today} \\
&\quad\text{Question: is this sentence grammatically correct?} \\
&\quad\text{Sentence: \_\_\_\_\_\_\_\_}
\end{align*}
$$

The above approach is a way to **"prompt"** the model without altering the weights. However, this requires that the model is able to recognize prompt structures such as questions or commands.One way to do this is to incorporate training samples with instructions and their corresponding responses into the pre-training dataset. While this method is straightforward, building and training LLMs from scratch is computationally expensive. Moreover, making instruction-following data effective for pre-training requires a significant amount of such data, but collecting large-scale labeled data for all tasks of interest is very difficult.

In practice, prompting is a huge topic so we will take a closer look in later notebooks.

## Fine-tuning
A second method, which has been a de facto standard in recent research, is to adapt LLMs via fine-tuning. As such, the token prediction ability learned in the pre-training phase can be generalized to accomplish new tasks. To achieve this, we can slightly fine-tune the model parameters using instruction-following data -- also known as **instruction fine-tuning**. The idea behind fine-tuning is that some general knowledge of language has been acquired in pre-training, but we need a mechanism to activate this knowledge for applying it to new tasks. An instruction fine-tuning sample can be represented by a tuple consisting of the input sequence and the desired output.

To acquire **instruction-follow up** abilities, a certain amount of fine-tuning data is required (although much less than pre-training) usually consisting of a diverse set of instruction response examples. It has been found that scaling the number of fine-tuning tasks is beneficial for improving the performance of LLMs.

Once we have prepared the fine-tuning samples, the process pretty much just follows the same process as encoders (and supervised training in general). Given the samples $D_{\text{tune}}$ and parameters $\hat{\theta}$ found via pre-training, we can modify the training objective to be
$$\tilde{\theta} = \arg\max_{\hat{\theta}_{\text{tune}}}\sum_{i\in D_{\text{tune}}} \ell_{\hat{\theta}_{\text{tune}}, i}$$
For each sample in the fine tuning dataset, we divide the input and the output segment where
$$\text{sample} = [x_{\text{sample}}, y_{\text{sample}}]$$
and the loss function is then calculated over the subsequence $y_\text{sample}$ rather than the entire sequence. In other words, we only compute loss for the part of the sequence that corresponds to the desired output, not the prefix.
$$ L_{\hat{\theta}_{\text{tune}}} = -\log \mathbb{P}_{\hat{\theta}_{\text{tune}}}(y_{\text{sample}} | x_{\text{sample}})$$
This also means that the forward pass works as usual, however, the error gradients are only propagated backwards through the parts of the network that correspond to the labels $y_{\text{sample}}$, leaving the rest of the network unchanged. A simple way to do this is to simply mask out the context tokens during gradient calculations. Even though the context tokens have no loss, their hidden states still influence the output tokens via attention. 

## Few-shot Learning
Generally speaking, the pre-training phase corresponds to a LLM state that has gained some sort of knowledge for understanding instructions and generating responses. However, these abilities are not fully activated until the fine-tuning phase, which requires some sort of supervision. The general instruction-following behavior emerges as we fine-tune the models with a relatively small amount of labeled data. As a result, we can achieve some level of **zero-shot learning**: the fine-tuned models can handle new tasks that they have not been explicitly trained or fine-tuned for.

## Hyperparameter checklist